In [6]:
!pip install duckduckgo-search

In [7]:
import duckduckgo_search
print(duckduckgo_search.__version__)


0.9.5


In [8]:
import os
print(os.listdir())


['.ipynb_checkpoints', 'downloaded_images', 'emiley_clarke_6.jpg', 'emiley_clarke_9.jpg', 'graph_visualization.png', 'image_1.jpg', 'Langgraph-agent-waether and tavily.ipynb', 'langgraph-agents-movie.ipynb', 'Langraph-image-webscrapping.ipynb', 'Rough.ipynb', 'Understanding-Langraph2.ipynb']


### Tool

In [13]:
# !pip install --upgrade duckduckgo-search


In [1]:
import duckduckgo_search
print(duckduckgo_search.__version__)


8.1.1


In [2]:
import os
import requests
import random
from duckduckgo_search import DDGS



from typing import Annotated
from langchain_core.tools import tool #tool 





In [3]:
@tool
def download_images(query: str,count: int,folder: str) -> str:
    
    """Searches DuckDuckGo for images and downloads them to the specified folder."""

    os.makedirs(folder, exist_ok=True)

    try:
        with DDGS() as ddgs:
            results = ddgs.images(keywords=query, max_results=count)
            for i, result in enumerate(results):
                url = result["image"]
                try:
                    response = requests.get(url, timeout=10)
                    if response.status_code == 200:
                        file_path = os.path.join(folder, f"{query.replace(' ', '_')}_{i+1}.jpg")
                        with open(file_path, "wb") as f:
                            f.write(response.content)
                    else:
                        print(f"Failed to download from {url}")
                except Exception as e:
                    print(f"Error downloading image {i+1}: {e}")
        return f"Downloaded {count} images for '{query}' in folder '{folder}'."
    except Exception as e:
        return f"Failed to download images due to: {str(e)}"


In [4]:
download_images.run({
    "query": "Lion",
    "count": 5,
    "folder": "downloaded_images"
})


C:\Users\esvit\AppData\Local\Temp\ipykernel_25296\1274580335.py:9: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


"Downloaded 5 images for 'Lion' in folder 'downloaded_images'."

### Model

In [5]:


from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="https://api.together.xyz/v1", 
    api_key=os.environ["TOGETHERAI_API_KEY"],
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
)

### Binding tool

In [6]:
#bind_tools to give model knowledge about these tools
tools = [download_images]
model_with_tools = model.bind_tools(tools)

In [7]:
model_with_tools.invoke('usa').content

''

### Model with agent

In [8]:
from langgraph.prebuilt import create_react_agent #agent
import warnings
warnings.filterwarnings("ignore")

In [9]:
prompt = f'''you are a helpful assistant that can help people to answer the questions.

        for general queries you use your pretrained knowledge to answer the questions .
        
        and whenevr user asks you to download images with a particular keyword then you 
        will use the download_images tool to scrap the images.refer to the example below.

        user:download images 5 of a cute puppies 
        then you will use cute puppies as a keyword and download it with the number of images ementiond.


        always store the images in a default folder of downloaded_images.
        
        
        Only use tool whenever user asks to for the download of images else do not use'''

In [10]:

agent = create_react_agent(model=model, tools=tools, prompt=prompt)

In [11]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print() 

inputs = {"messages": [("user", "hi")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

hi
================================== Ai Message ==================================

I'm here to help. How can I assist you today?


In [12]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print() 

inputs = {"messages": [("user", "recommedn me 5 best places to visit in dubai")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

recommedn me 5 best places to visit in dubai
================================== Ai Message ==================================

You haven't asked me to download any images. I can provide general information about the best places to visit in Dubai. 

If you want to download images of these places, please let me know and I can assist you with that.


In [13]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print() 

inputs = {"messages": [("user", "Download 10 images of Falcon bird")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

Download 10 images of Falcon bird
================================== Ai Message ==================================
Tool Calls:
  download_images (call_h2ud7bjyhqf1c9lypfcs59jp)
 Call ID: call_h2ud7bjyhqf1c9lypfcs59jp
  Args:
    count: 10
    folder: downloaded_images
    query: Falcon bird


C:\Users\esvit\AppData\Local\Temp\ipykernel_25296\1274580335.py:9: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Failed to download from https://www.birdsandblooms.com/wp-content/uploads/2023/05/306980241_1_Rodney_Hendrickson_BNB_PC_2022.jpg
================================= Tool Message =================================
Name: download_images

Downloaded 10 images for 'Falcon bird' in folder 'downloaded_images'.
================================== Ai Message ==================================

The images of Falcon bird have been downloaded and saved in the "downloaded_images" folder. Is there anything else I can help you with?


In [14]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print() 

inputs = {"messages": [("user", "Download 10 images of Polar bear")]}

print_stream(agent.stream(inputs, stream_mode="values"))  # Ensure agent is properly defined


================================ Human Message =================================

Download 10 images of Polar bear
================================== Ai Message ==================================
Tool Calls:
  download_images (call_vyz6sgcj3cj16qab9exuf1o4)
 Call ID: call_vyz6sgcj3cj16qab9exuf1o4
  Args:
    count: 10
    folder: downloaded_images
    query: Polar bear


C:\Users\esvit\AppData\Local\Temp\ipykernel_25296\1274580335.py:9: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Failed to download from https://www.thoughtco.com/thmb/FcM6FJ1e6wrimHA7zTdAArfRwjw=/1500x0/filters:no_upscale():max_bytes(150000):strip_icc()/polar-bear-150005875-5c4db42046e0fb0001a8e7b8.jpg
================================= Tool Message =================================
Name: download_images

Downloaded 10 images for 'Polar bear' in folder 'downloaded_images'.
================================== Ai Message ==================================

I have downloaded 10 images of Polar bears in the downloaded_images folder.
